In [17]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error


In [12]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
#df["Date"] = pd.to_datetime(df["Date"])
# dont make timeseries data for xgboost

#df.set_index("Date", inplace=True)
df = df.reset_index(drop=True)
df.head(6)

,Date,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
0,2001-07-31,3.67,3.54,3.47,3.53,3.79,4.06,4.57,4.86,5.07,5.61,5.51
1,2001-08-01,3.65,3.53,3.47,3.56,3.83,4.09,4.62,4.90,5.11,5.63,5.53
2,2001-08-02,3.65,3.53,3.46,3.57,3.89,4.17,4.69,4.97,5.17,5.68,5.57
3,2001-08-03,3.63,3.52,3.47,3.57,3.91,4.22,4.72,4.99,5.20,5.70,5.59
4,2001-08-06,3.62,3.52,3.47,3.56,3.88,4.17,4.71,4.99,5.19,5.70,5.59
5,2001-08-07,3.63,3.52,3.47,3.56,3.90,4.19,4.72,5.00,5.20,5.71,5.60


In [15]:
#seperate into different dataframes
maturities = []
matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

for col in df.columns:
    maturities.append(df[col])

#now create a table for each maturity that has 5 columns, the lag values for 1-20 days

tables = []
features = ['lag1', 'lag2', 'lag3', 'lag4', 'lag5']
# one table for each maturity
for mat in matList:
    
    # table 5 maturities, 
    # ,'6','7','8','9','10', '11','12','13','14','15', '16','17','18','19','20', '21'
    rows = []
    
    for i in range(4, len(df.index) - 5):
            #adds in all 20 lag values for each maturity
        lag1 = df[mat].iloc[i]
        lag2 = df[mat].iloc[i-1]
        lag3 = df[mat].iloc[i-2]
        lag4 = df[mat].iloc[i-3]
        lag5 = df[mat].iloc[i-4]

        #target is value 5 days in advance, so this model is 5 day horizon
        target = df[mat].iloc[i+5]
        rows.append([lag1, lag2, lag3, lag4, lag5, target])

    df1 = pd.DataFrame(rows, columns=["lag1", "lag2", "lag3", "lag4", "lag5", "target"])            
    tables.append(df1)


In [18]:
xy = []

for tab in tables:
    x = tab[features]
    y = tab["target"]
    xy.append((x, y))

x_train = []
y_train = []
x_test = []
y_test = []

for x, y in xy:
    #split into train and test
    split = int(len(x) * 0.8)
    x_train.append(x[:split])
    y_train.append(y[:split])
    x_test.append(x[split:])
    y_test.append(y[split:])

models = []
for i in range(len(x_train)):
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators = 500,
        learning_rate = 0.02,
        max_depth = 3,
        subsample = 0.8,
        colsample_bytree = 0.8, 
        reg_lambda = 5, 
        random_state = 7
    )
    model.fit(x_train[i], y_train[i])
    models.append(model)

predictions = []
for i in range(len(models)):
    pred = models[i].predict(x_test[i])
    predictions.append(pred)

rmse = []
for i in range(len(predictions)):
    rmse.append(root_mean_squared_error(y_test[i], predictions[i]))

for i in range(len(matList)):
    print(f"Maturity: {matList[i]}, RMSE: {rmse[i]}")



Maturity: 0Y1M, RMSE: 0.2631388558384961
Maturity: 0Y3M, RMSE: 0.21366971230666887
Maturity: 0Y6M, RMSE: 0.1440845622105307
Maturity: 1Y, RMSE: 0.14045235820717517
Maturity: 2Y, RMSE: 0.15372563174668002
Maturity: 3Y, RMSE: 0.1529158240583423
Maturity: 5Y, RMSE: 0.1502993214026844
Maturity: 7Y, RMSE: 0.1456655563705179
Maturity: 10Y, RMSE: 0.13605565294183078
Maturity: 20Y, RMSE: 0.12343143783940384
Maturity: 30Y, RMSE: 0.12418931810724353
